In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import *
import sys
sys.path.append("/Workspace/Users/sivana9908_gmail.com#ext#@sivana9908gmail.onmicrosoft.com/Uber-Eats-End-to-End-_Azure-Data-Engineering-Project")
from src.common.spark_utils import standardize_columns
from delta.tables import DeltaTable

#read data from adls gen2

In [0]:

df_res = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://bronze@ubereaststorage.dfs.core.windows.net/sql/restaurants/")
)
display(df_res)


In [0]:
# standadize column names

df_res= standardize_columns(df_res)

# standadize data types
df_res = df_res.withColumn("restaurant_id",col("restaurant_id").cast("int"))\
                  .withColumn("double", col("rating").cast("double"))\
                      .withColumn("created_at", col("created_at").cast("timestamp"))\
                          .withColumn("updated_at", col("updated_at").cast("timestamp"))

#standadize values
df_res = df_res.withColumn("status", upper(trim(col("status"))))

#handling nulls
df_res  = df_res.filter(col("restaurant_id").isNotNull())

# deduplication means
df_res = df_res.dropDuplicates(["restaurant_id"])
#apply the business rules
df_res = df_res.filter(col("status").isin("open","active"))\
                .filter(col("restaurant_id").isNotNull())

#validation
print(f"schema")
df_res.printSchema()


dedup= df_res.groupBy("restaurant_id").count().filter(col("count") >1).count()
print("duplication rows",dedup)

businesskeynulls= df_res.filter(col("restaurant_id").isNull()).count()
print("null values in business key",businesskeynulls)

invalid_status = df_res.filter(~col("status").isin("open","active")).count()
print("invalid status",invalid_status)

               








In [0]:
table_name= "ubereats_databricks.silver.silver_restaurants"

if not spark.catalog.tableExists(table_name):

     df_res .write.format("delta").mode("overwrite").saveAsTable(table_name)   
      
else:

    target = DeltaTable.forName(spark, table_name)
    target.alias("t") \
        .merge(
            df_res.alias("s"),
            "t.restaurant_id = s.restaurant_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
